# Mean Difference Map (Figure 4)

Computes the per-pixel difference in month-stratified mean AdDSWE class between the
pre-2011 and post-2012 epochs, and builds the **Figure 4** panels.

- **Produces:** `analysis_AdDSWE_MeanDifference_PostMinusPre_MonthStratified.tif`
  (7-band; Dryad `inundation/analysis/pixel_difference/`) and Figure 4.
- **Inputs:** AdDSWE monthly products (EE assets) for the difference computation; the
  exported difference GeoTIFF for the figure panels.

> The generation section requires Earth Engine; the figure section runs on the exported
> GeoTIFF. Paths are set within the notebook.


In [ ]:
import os

# =============================== PATH CONFIGURATION ===============================
# Point DRYAD_ROOT at your local copy of the Dryad archive (doi:10.5061/dryad.msbcc2gct).
DRYAD_ROOT = r"."            # e.g. r"E:\Okavango\Data\For_Dryad"
OUTPUT_DIR = r"./outputs"    # local folder for derived CSVs / figures
os.makedirs(OUTPUT_DIR, exist_ok=True)


# AdDSWE Mean Difference Analysis (Figure 4)

End-to-end pipeline for the month-stratified post-minus-pre AdDSWE difference
map. **Part I (§1–§8)** computes and exports the difference raster in Google
Earth Engine. **Part II (§9–§14)** renders the publication figure panels from
the downloaded GeoTIFF.

**Periods** — Pre: June 1984 – May 2010 · Post: March 2013 – December 2025,
bracketing the 2010–2012 La Niña and the associated data gap, consistent with
the pre/post definitions used in the divergence and change-point analyses.

**Why month-stratified?** Monthly scene availability is irregular (calendar-month
shares of 6–10% in the pre-period vs. a uniform 8.3%). A flat mean over all
scenes would therefore weight some seasons more than others, confounding a real
regime change with a sampling artifact, and epochs that start or end mid water
year would add a partial-year version of the same problem. Stratifying —
averaging each calendar month first, then averaging the twelve monthly means
with equal weight — gives every month exactly 1/12 weight, eliminating both
issues by construction.

**Workflow**: run §1–§8 (GEE export job) → download the COG from GCS to the
local path in §9 → run §9–§14 → assemble panels in Affinity Designer.

---
## Part I · Raster generation (Google Earth Engine)

### §1 · Imports and Earth Engine initialization

Original module docstring retained below for provenance.

In [ ]:
#!/usr/bin/env python3
"""
Mean AdDSWE Difference Map: Post-Break minus Pre-Break (Month-Stratified)
==========================================================================
For every 30m pixel in the Okavango Delta, computes the MONTH-STRATIFIED
mean DSWE classification value in two sub-periods:

    Pre:  June 1984 – May 2010
    Post: March 2013 – December 2025

Outputs a single-band difference raster (post_mean − pre_mean) where:
    Positive values → pixel became wetter (higher mean DSWE class)
    Negative values → pixel became drier  (lower mean DSWE class)

Stratification: within each period, all available images for each calendar
month are averaged first (mean of all Januaries, all Februaries, ...); the
period mean is then the equal-weight average of the 12 monthly means. Every
calendar month therefore carries exactly 1/12 weight regardless of how many
scenes it contributes, removing seasonal sampling bias from irregular
monthly coverage and rendering partial water years at the epoch edges
irrelevant.

Also exports the stratified pre-mean and post-mean as separate bands, the
total valid-observation count per period, and a months_covered band per
period (0-12: how many calendar months have >=1 valid observation at that
pixel; 12 in both periods is required for inclusion in the difference band).

Exports:
    1. Cloud-Optimized GeoTIFF to gs://okavango-addswe-products/
    2. Interactive map visualization in the notebook

Usage (Jupyter):
    Just run the cell — no CLI arguments needed.
"""

import ee
import geemap

# ── INITIALIZE ───────────────────────────────────────────────────────────────
ee.Initialize(project='ee-okavango')

### §2 · Configuration

Period boundaries, asset locations, and export settings. The export filename
carries a `_MonthStratified` suffix so the original (unstratified) product is
not overwritten — keep both until the side-by-side comparison (§10) is done.

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────

# Asset paths
DSWE_COLLECTION = 'projects/ee-okavango/assets/water_masks/monthly_DSWE_Landsat_30m_v4/DSWE_Products'
STUDY_AREA      = 'projects/phd-work2023/assets/Okavango_StudAr'

# GCS export
GCS_BUCKET = 'okavango-addswe-products'
EXPORT_FILENAME = 'AdDSWE_MeanDifference_PostMinusPre_MonthStratified'

# Sub-period boundaries — matches Chow / MK framework
PRE_START  = (1984, 6)
PRE_END    = (2010, 5)
POST_START = (2013, 3)
POST_END   = (2025, 12)

# CRS and resolution
CRS = 'EPSG:32734'  # UTM Zone 34S
SCALE = 30

### §3 · Study area

In [ ]:
# ── LOAD STUDY AREA ─────────────────────────────────────────────────────────
study_area = ee.FeatureCollection(STUDY_AREA).geometry()

### §4 · Build the monthly image collections

One image per available calendar month in each period, each tagged with a
`month` property used by the stratification in §5.

In [ ]:
# ── BUILD IMAGE COLLECTIONS FOR EACH PERIOD ──────────────────────────────────

def ym_to_int(year, month):
    """Convert (year, month) to integer YYYYMM for comparison."""
    return year * 100 + month


def list_existing_assets(collection_path):
    """
    List all image asset IDs that actually exist in a GEE image collection.
    Returns a set of asset IDs for fast lookup.
    """
    existing = set()
    page_token = None
    while True:
        params = {'parent': collection_path, 'pageSize': 500}
        if page_token:
            params['pageToken'] = page_token
        result = ee.data.listAssets(params)
        for asset in result.get('assets', []):
            existing.add(asset['id'])
        page_token = result.get('nextPageToken')
        if not page_token:
            break
    return existing


def build_period_collection(start, end, existing_assets):
    """
    Build an ee.ImageCollection of monthly DSWE images within a
    (year, month) range, only including assets that actually exist.

    Parameters
    ----------
    start : tuple (year, month)
    end   : tuple (year, month)
    existing_assets : set
        Set of asset ID strings returned by list_existing_assets()

    Returns
    -------
    ee.ImageCollection
    """
    images = []
    for year in range(start[0], end[0] + 1):
        for month in range(1, 13):
            ym = ym_to_int(year, month)
            if ym < ym_to_int(*start) or ym > ym_to_int(*end):
                continue
            asset_id = f'{DSWE_COLLECTION}/DSWE_{year}_{month:02d}'
            if asset_id not in existing_assets:
                continue
            img = (ee.Image(asset_id)
                   .select(0)
                   .rename('dswe')
                   .set('year', year)
                   .set('month', month)
                   .set('ym', ym))
            images.append(img)
    return ee.ImageCollection(images)


print('Listing existing DSWE assets...')
existing_assets = list_existing_assets(DSWE_COLLECTION)
print(f'  Found {len(existing_assets)} total DSWE assets')

print('Building image collections...')
pre_col  = build_period_collection(PRE_START, PRE_END, existing_assets)
post_col = build_period_collection(POST_START, POST_END, existing_assets)

print(f'  Pre  collection: {pre_col.size().getInfo()} images')
print(f'  Post collection: {post_col.size().getInfo()} images')

### §5 · Month-stratified period means

The core methodological step. For each period: (1) average all available
images within each calendar month (per-pixel; cloud-masked pixels are skipped),
then (2) average the twelve monthly means with equal weight. `months_covered`
(0–12) records, per pixel, how many calendar months have ≥1 valid observation.

In [ ]:
# ── COMPUTE MONTH-STRATIFIED MEAN AND COVERAGE PER PERIOD ────────────────────────────────────────

def stratified_mean(col, prefix):
    """Month-stratified period mean.

    1. For each calendar month m, average all images in `col` having that
       month (per-pixel; cloud-masked pixels are skipped by .mean()).
    2. Average the 12 monthly-mean images with equal weight.

    Returns (stratified_mean, months_covered), where months_covered counts,
    per pixel, how many of the 12 calendar months have >=1 valid
    observation. Pixels with months_covered < 12 are NOT masked here --
    masking is applied to the difference band below, so the per-period
    bands remain inspectable.
    """
    monthly_means = []
    for m in range(1, 13):
        mcol = col.filter(ee.Filter.eq('month', m))
        monthly_means.append(mcol.mean().set('month', m))
    mm = ee.ImageCollection(monthly_means)
    strat = mm.mean().rename(f'{prefix}_mean')
    months_covered = mm.count().rename(f'{prefix}_months_covered')
    return strat, months_covered


print('Computing month-stratified per-period means...')

pre_mean,  pre_months  = stratified_mean(pre_col,  'pre')
post_mean, post_months = stratified_mean(post_col, 'post')

pre_mean,  pre_months  = pre_mean.clip(study_area),  pre_months.clip(study_area)
post_mean, post_months = post_mean.clip(study_area), post_months.clip(study_area)

# Total observation counts (diagnostic; same definition as previous version)
pre_count  = pre_col.count().rename('pre_count').clip(study_area)
post_count = post_col.count().rename('post_count').clip(study_area)

### §6 · Difference band with strict coverage mask

The difference is computed only where **all 12 calendar months are represented
in both periods**. Without this mask, a pixel missing (say) Decembers in the
pre-period would silently average over 11 months — reintroducing exactly the
seasonal imbalance the stratification removes.

In [ ]:
# ── DIFFERENCE MAP ────────────────────────────────────────

# Strict stratification: require all 12 calendar months represented in
# BOTH periods; otherwise unequal month sets would silently reintroduce
# the seasonal sampling bias the stratification removes.
full_coverage = pre_months.eq(12).And(post_months.eq(12))

diff = (post_mean.subtract(pre_mean)
        .updateMask(full_coverage)
        .rename('difference'))

### §7 · Assemble the multi-band output

Band order: 1 difference · 2 pre_mean · 3 post_mean · 4 pre_count ·
5 post_count · 6 pre_months_covered · 7 post_months_covered.

In [ ]:
# ── COMBINE INTO MULTI-BAND OUTPUT ────────────────────────────────────────

output = (diff
          .addBands(pre_mean)
          .addBands(post_mean)
          .addBands(pre_count)
          .addBands(post_count)
          .addBands(pre_months)
          .addBands(post_months)
          .toFloat())

# Band order: difference, pre_mean, post_mean, pre_count, post_count,
#             pre_months_covered, post_months_covered

### §8 · Export to Google Cloud Storage

After the task completes, download the COG to the local `TIF_PATH` in §9
(e.g. via `gsutil cp`). The optional preview map renders the difference layer
in-notebook before committing to the export.

In [ ]:
# ── EXPORT TO GCS ────────────────────────────────────────────────────────────

print('Starting export to GCS...')

task = ee.batch.Export.image.toCloudStorage(
    image=output,
    description=EXPORT_FILENAME,
    bucket=GCS_BUCKET,
    fileNamePrefix=f'analysis/{EXPORT_FILENAME}',
    region=study_area,
    crs=CRS,
    scale=SCALE,
    maxPixels=1e10,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True},
)
task.start()
print(f'  ✓ Export task submitted: {EXPORT_FILENAME}')
print(f'    Destination: gs://{GCS_BUCKET}/analysis/{EXPORT_FILENAME}.tif')
print(f'    Monitor at: https://code.earthengine.google.com/tasks')

In [ ]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────

print('\nBuilding map visualization...')

Map = geemap.Map()
Map.centerObject(study_area, zoom=10)

# Difference layer — diverging blue-white-red
diff_vis = {
    'min': -1.0,
    'max':  1.0,
    'palette': [
        '#67001f', '#b2182b', '#d6604d', '#f4a582', '#fddbc7',  # drier (red)
        '#f7f7f7',                                                # no change
        '#d1e5f0', '#92c5de', '#4393c3', '#2166ac', '#053061',  # wetter (blue)
    ],
}
Map.addLayer(diff, diff_vis, 'Difference (Post − Pre, month-stratified)', True)

# Pre-period mean
mean_vis = {
    'min': 0,
    'max': 3.0,
    'palette': ['#f7fbff', '#c6dbef', '#6baed6', '#2171b5', '#08306b'],
}
Map.addLayer(pre_mean, mean_vis, 'Pre-period Mean DSWE', False)

# Post-period mean
Map.addLayer(post_mean, mean_vis, 'Post-period Mean DSWE', False)

# Study area outline
Map.addLayer(study_area, {'color': '000000'}, 'Study Area', True, opacity=0.5)

Map.addLayerControl()

print('✓ Map ready.')
Map

---
## Part II · Figure panels (local rendering)

Reads band 1 (difference) of the downloaded GeoTIFF and produces panel a)
(full delta with bounding-box overlays) plus one zoomed inset per box
(b, c, d, … sorted north → south), all saved individually for manual
assembly in Affinity Designer.

### §9 · Local imports, paths, and fonts

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1 import make_axes_locatable
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.windows import from_bounds
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import box as shapely_box


# ── LOCAL PATHS ──────────────────────────────────────────────────────────────

TIF_PATH   = os.path.join(DRYAD_ROOT, "inundation", "analysis", "pixel_difference", "analysis_AdDSWE_MeanDifference_PostMinusPre_MonthStratified.tif")
SHP_STUDY  = os.path.join(DRYAD_ROOT, "study_areas", "Okavango_StudAr_OSM_v2.shp")
OUTPUT_DIR = OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)


def setup_fonts():
    """Register Arial if available; fall back gracefully."""
    arial_paths = [
        '/usr/share/fonts/truetype/msttcorefonts/Arial.ttf',
        '/usr/share/fonts/truetype/arial.ttf',
        'C:/Windows/Fonts/arial.ttf',
    ]
    for p in arial_paths:
        if os.path.isfile(p):
            fm.fontManager.addfont(p)
            break
    plt.rcParams.update({
        'font.family': 'Arial',
        'font.size': 8,
        'svg.fonttype': 'none',
        'pdf.fonttype': 42,
    })


setup_fonts()

### §10 · Color scheme and small helpers

Diverging Brown–Teal (BrBG): brown = drier post-shift, teal = wetter
post-shift, near-white = no change. Symmetric limits are set in §11 from the
99th percentile of |Δ|.

In [ ]:
CMAP_COLORS = [
    '#8C510A', '#BF812D', '#D8B365', '#F6E8C3',
    '#F5F5F5',
    '#C7EAE5', '#80CDC1', '#5AB4AC', '#01665E',
]

def make_cmap():
    return mcolors.LinearSegmentedColormap.from_list('BrBG_custom', CMAP_COLORS, N=256)

# Box outline color
BOX_COLOR = '#000000'   # Black
BOX_LW    = 1.5


def utm_to_lonlat(x, y, src_crs):
    transformer = Transformer.from_crs(src_crs, 'EPSG:4326', always_xy=True)
    return transformer.transform(x, y)

def lonlat_to_utm(lon, lat, dst_crs):
    transformer = Transformer.from_crs('EPSG:4326', dst_crs, always_xy=True)
    return transformer.transform(lon, lat)

def format_lon(val):
    return f'{val:.1f}\u00b0E' if val >= 0 else f'{abs(val):.1f}\u00b0W'

def format_lat(val):
    return f'{abs(val):.1f}\u00b0S' if val < 0 else f'{val:.1f}\u00b0N'


def add_north_arrow(ax, x=0.92, y=0.95, size=0.06):
    """
    Draw a simple north arrow on the axes.

    Parameters
    ----------
    ax : matplotlib Axes
    x, y : float
        Position in axes coordinates (0–1).
    size : float
        Arrow length in axes-coordinate units.
    """
    ax.annotate(
        'N',
        xy=(x, y - size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        fontsize=8, fontname='Arial', fontweight='bold',
        ha='center', va='center',
        arrowprops=dict(
            arrowstyle='->', color='black', lw=1.5,
        ),
    )

### §11 · Load, clip, and diagnose the raster

Loads the difference band clipped to the study area, sets symmetric color
limits, and reports the **coverage diagnostic**: the share of study-area pixels
excluded by the strict 12/12-month rule (expected to be small — every calendar
month has ≥11 scenes per period, so exclusions come only from persistent
per-pixel cloud masking). A warning fires if the file has <7 bands, i.e. the
old unstratified product.

In [ ]:
def load_clipped_diff(tif_path, shp_path):
    """Load difference band (band 1) clipped to the study area polygon.

    Also returns the full clipped band stack so the caller can run
    coverage diagnostics on the months_covered bands (6-7) when present.
    """
    gdf = gpd.read_file(shp_path)
    with rasterio.open(tif_path) as src:
        src_crs = src.crs
        if gdf.crs != src_crs:
            gdf = gdf.to_crs(src_crs)
        shapes = list(gdf.geometry)
        clipped, clipped_transform = rio_mask(
            src, shapes, crop=True, nodata=np.nan, filled=True
        )
        diff = clipped[0]
        nrows, ncols = diff.shape
        left   = clipped_transform.c
        top    = clipped_transform.f
        right  = left + ncols * clipped_transform.a
        bottom = top  + nrows * clipped_transform.e
    return diff, [left, right, bottom, top], src_crs, gdf, clipped_transform, clipped


def read_diff_window(tif_path, bounds_utm, src_crs):
    """Read the difference band for a specific UTM bounding box."""
    with rasterio.open(tif_path) as src:
        window = from_bounds(
            bounds_utm[0], bounds_utm[2],  # left, bottom
            bounds_utm[1], bounds_utm[3],  # right, top
            src.transform
        )
        diff = src.read(1, window=window)
        win_transform = src.window_transform(window)
        nrows, ncols = diff.shape
        left   = win_transform.c
        top    = win_transform.f
        right  = left + ncols * win_transform.a
        bottom = top  + nrows * win_transform.e
    return diff, [left, right, bottom, top]


# ── LOAD + DIAGNOSE ──────────────────────────────────────────────────────────

print('Loading difference raster...')
diff, extent_utm, src_crs, study_gdf, _, stack = load_clipped_diff(TIF_PATH, SHP_STUDY)
valid = diff[~np.isnan(diff)]
print(f'  Clipped raster value range: {valid.min():.3f} to {valid.max():.3f}')

if stack.shape[0] >= 7:
    pre_mean_band = stack[1]
    in_study = ~np.isnan(pre_mean_band)
    masked_by_rule = in_study & np.isnan(diff)
    n_in, n_masked = int(in_study.sum()), int(masked_by_rule.sum())
    if n_in > 0:
        print(f'  Coverage mask: {n_masked:,} of {n_in:,} study-area pixels '
              f'({100.0 * n_masked / n_in:.2f}%) lack all-12-month coverage '
              f'in one or both periods and are excluded.')
else:
    print('  NOTE: input raster has < 7 bands — this looks like the '
          'unstratified (v1) product. Confirm TIF_PATH.')

# Symmetric color limits (99th percentile of |Δ|, rounded up to 0.1)
vmax = np.ceil(np.nanpercentile(np.abs(valid), 99) * 10) / 10
vmin = -vmax
print(f'  Color range: {vmin:.1f} to {vmax:.1f}')

### §12 · Bounding boxes for the insets

Sorted north → south; labels b), c), d), … (panel a is the main map).

In [ ]:
print('Loading bounding boxes...')
# Inset-panel bounding boxes (WGS84 / EPSG:4326), hardcoded so no external
# shapefile is needed. Each tuple is (min_lon, min_lat, max_lon, max_lat).
_INSET_BOXES = [
    (22.70306575, -19.04995249, 22.93437098, -18.94155971),
    (21.96191417, -18.43677492, 21.97865517, -18.42060093),
    (23.03687228, -18.78670115, 23.12612842, -18.69100598),
    (22.32989235, -19.14763501, 22.34782462, -19.13137074),
    (22.23294178, -19.27396614, 22.24334184, -19.26438191),
    (23.58628958, -19.78665667, 23.66686071, -19.70689753),
]
boxes_gdf = gpd.GeoDataFrame(
    geometry=[shapely_box(*_b) for _b in _INSET_BOXES], crs="EPSG:4326"
)
if boxes_gdf.crs != src_crs:
    boxes_gdf = boxes_gdf.to_crs(src_crs)

boxes_gdf['centroid_y'] = boxes_gdf.geometry.centroid.y
boxes_gdf = boxes_gdf.sort_values('centroid_y', ascending=False).reset_index(drop=True)

n_boxes = len(boxes_gdf)
box_labels = [f'{chr(ord("b") + i)})' for i in range(n_boxes)]

print(f'  Found {n_boxes} bounding boxes (sorted N → S):')
for i, (_, row) in enumerate(boxes_gdf.iterrows()):
    b = row.geometry.bounds
    print(f'    {box_labels[i]}  {(b[2]-b[0])/1000:.1f} × {(b[3]-b[1])/1000:.1f} km')

### §13 · Panel a) — full delta difference map

In [ ]:
def plot_main_map(diff, extent_utm, src_crs, study_gdf, boxes_gdf,
                  box_labels, vmin, vmax, output_dir):
    """
    Full delta difference map with bounding box overlays.
    """
    cmap = make_cmap()

    fig, ax = plt.subplots(figsize=(6.5, 8.0))

    im = ax.imshow(
        diff, extent=extent_utm, cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation='nearest', aspect='equal',
    )

    # Study area boundary
    study_gdf.boundary.plot(ax=ax, color='black', linewidth=0.6, alpha=0.6)

    # Bounding boxes (no labels — added manually in Affinity)
    for idx, row in boxes_gdf.iterrows():
        bounds = row.geometry.bounds  # (minx, miny, maxx, maxy)
        w = bounds[2] - bounds[0]
        h = bounds[3] - bounds[1]
        rect = mpatches.FancyBboxPatch(
            (bounds[0], bounds[1]), w, h,
            boxstyle='square,pad=0',
            linewidth=BOX_LW, edgecolor=BOX_COLOR,
            facecolor='none', zorder=10,
        )
        ax.add_patch(rect)

    # Remove all axes, ticks, and border
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Scale bar
    scalebar = ScaleBar(
        1, units='m', location='lower right', length_fraction=0.18,
        font_properties={'family': 'Arial', 'size': 7},
        box_alpha=0.7, color='black', box_color='white',
    )
    ax.add_artist(scalebar)

    # Colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='3%', pad=0.08)
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label('\u0394 mean AdDSWE class (post \u2212 pre 2010\u20132012 La Ni\u00f1a)',
                   fontname='Arial', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    for lbl in cbar.ax.get_yticklabels():
        lbl.set_fontname('Arial')

    plt.tight_layout()

    # Save
    base = 'AdDSWE_MeanDiff_MainMap'
    for ext in ['png', 'jpg', 'svg', 'pdf']:
        path = os.path.join(output_dir, f'{base}.{ext}')
        dpi = 1000 if ext in ['png', 'jpg'] else None
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        print(f'  \u2713 Saved: {path}')

    plt.show()
    plt.close(fig)


plot_main_map(diff, extent_utm, src_crs, study_gdf, boxes_gdf,
              box_labels, vmin, vmax, OUTPUT_DIR)

### §14 · Inset panels b), c), d), …

Each inset re-reads its window directly from the GeoTIFF at the identical
color scale; no ticks or colorbars (labels added in Affinity).

In [ ]:
def plot_inset(tif_path, bounds_utm, label, vmin, vmax, src_crs,
               output_dir):
    """
    Single inset map for one bounding box.
    No lat/lon ticks, no colorbar, no scale bar. North arrow only.
    Labels added manually in Affinity Designer.
    """
    cmap = make_cmap()

    diff, extent = read_diff_window(tif_path, bounds_utm, src_crs)

    # Determine figure size to maintain aspect ratio
    w_km = (extent[1] - extent[0]) / 1000
    h_km = (extent[3] - extent[2]) / 1000
    aspect = h_km / w_km
    fig_w = 4.0
    fig_h = fig_w * aspect

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        diff, extent=extent, cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation='nearest', aspect='equal',
    )

    # Remove axis ticks and border
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()

    # Save
    label_clean = label.replace(')', '').replace(' ', '')
    base = f'AdDSWE_MeanDiff_Inset_{label_clean}'
    for ext in ['png', 'jpg', 'svg', 'pdf']:
        path = os.path.join(output_dir, f'{base}.{ext}')
        dpi = 1000 if ext in ['png', 'jpg'] else None
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        print(f'  \u2713 Saved: {path}')

    plt.show()
    plt.close(fig)


for i, (_, row) in enumerate(boxes_gdf.iterrows()):
    label = box_labels[i]
    b = row.geometry.bounds
    bounds_utm = [b[0], b[2], b[1], b[3]]   # [left, right, bottom, top]
    print(f'\nGenerating inset {label}...')
    plot_inset(TIF_PATH, bounds_utm, label, vmin, vmax, src_crs, OUTPUT_DIR)

print(f'\n✓ All panels complete: {1 + n_boxes} figures in {OUTPUT_DIR}')

---
### Provenance and methods note

Period means are month-stratified: all available scenes within each calendar
month were averaged first, and the twelve monthly means were then averaged
with equal weight, ensuring uniform seasonal weighting despite irregular
monthly scene availability; pixels lacking valid observations in all twelve
calendar months in both periods were excluded from the difference band.
Displayed values are differences of mean AdDSWE class (an ordinal wetness
index); magnitudes are not interpreted quantitatively, and all statistical
claims in the manuscript rest on the month-normalized z-score framework.